# Validation Test Evaluation - Static droplet on slip wall (convergence study)

This notebook and the corresponding simulation setup notebook (StaticDropletOnSlipWall_convergence_Run.ipynb) are part of published results (cf. 5.1.2) found in *The extended discontinuous Galerkin method adapted for moving contact line problems via the generalized Navier boundary condition* (https://doi.org/10.1002/fld.5016).

In [ ]:
#r "BoSSSpad.dll"
//#r "..\..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [2]:
wmg.Init("CLpaper_StaticDropletOnSlipWall");
//OpenOrCreateDatabase(@"\\dc3\userspace\smuda\hpccluster\bkup-2025Oct29_000000.CLpaper_StaticDropletOnSlipWall");

Project name is set to 'CLpaper_StaticDropletOnSlipWall'.
Default Execution queue is chosen for the database.
trying to run from backup database...
   Bkup Database dirs: \\dc3\userspace\smuda\hpccluster\bkup-2025Oct29_000000.CLpaper_StaticDropletOnSlipWall;
Selecting newest available backup: \\dc3\userspace\smuda\hpccluster\bkup-2025Oct29_000000.CLpaper_StaticDropletOnSlipWall
Opening existing database '\\dc3\userspace\smuda\hpccluster\bkup-2025Oct29_000000.CLpaper_StaticDropletOnSlipWall'.


In [3]:
using System.IO;
using BoSSS.Foundation.Quadrature;
using BoSSS.Solution.XNSECommon;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [4]:
var sessions = wmg.Sessions;
//var sessions = databases.Pick(0).Sessions;
sessions

#0: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta5_cAstat90_ConvStudy2_k4_mesh5	6/24/2020 11:51:00 PM	7dc76cac...
#1: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta5_cAstat90_ConvStudy2_k3_mesh5	6/24/2020 11:49:40 PM	d684627f...
#2: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta5_cAstat90_ConvStudy2_k4_mesh4	6/24/2020 11:50:50 PM	03a654c3...
#3: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k4_mesh5	6/24/2020 11:35:29 PM	1561f038...
#4: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta5_cAstat90_ConvStudy2_k2_mesh5	6/24/2020 11:48:13 PM	194747d1...
#5: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta5_cAstat90_ConvStudy2_k4_mesh3	6/24/2020 11:50:38 PM	17c2169e...
#6: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta5_cAstat90_ConvStudy2_k3_mesh4	6/24/2020 11:49:14 PM	991e4c89...
#7: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta5_cAstat90_ConvS

In [5]:
// foreach (var s in sessions) {
//     s.ProjectName = "CLpaper_StaticDropletOnSlipWall";
//     s.KeysAndQueries["ProjectName"] = "CLpaper_StaticDropletOnSlipWall";
//     s.KeysAndQueries["SessionName"] = s.Name;
// }

In [6]:
//sessions

## Convergence study

In [7]:
int[] cAstatS = { 120, 90 };
int[] betaS = { 5, 0 };
int[] pDegS = { 2, 3, 4 };
int[] Meshes = { 0, 1, 2, 3, 4, 5 };

In [8]:
List<ISessionInfo> evalSess = new List<ISessionInfo>();

In [9]:
foreach (int cA in cAstatS) {
foreach (int beta in betaS) {
foreach (int pDeg in pDegS) {
foreach (int mesh in Meshes) {
    string studyName = $"StaticDropletOnWall_90Deg_beta{beta}_cAstat{cA}_ConvStudy2_k{pDeg}_mesh{mesh}";
    Console.WriteLine("Searching for: " + studyName);
    var studySess = sessions.Where(sess => sess.Name.Contains(studyName));
    int studyCount = studySess.Count();
    if (studyCount == 0) {
        Console.WriteLine("Not found!");
    } else if (studyCount > 1) {
        Console.WriteLine("Restart session found! Take last run");       
        evalSess.Add(studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single());
    } else {
        evalSess.Add(studySess.Single());
        Console.WriteLine("found");
    }
}
}
}
}
evalSess

Searching for: StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k2_mesh0
found
Searching for: StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k2_mesh1
found
Searching for: StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k2_mesh2
found
Searching for: StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k2_mesh3
found
Searching for: StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k2_mesh4
found
Searching for: StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k2_mesh5
found
Searching for: StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k3_mesh0
found
Searching for: StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k3_mesh1
found
Searching for: StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k3_mesh2
found
Searching for: StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k3_mesh3
found
Searching for: StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k3_mesh4
found
Searching for: StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k3_mesh5
found
Sear

#0: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k2_mesh0	6/24/2020 11:30:56 PM	ba5e832a...
#1: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k2_mesh1	6/24/2020 11:30:56 PM	0800ff38...
#2: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k2_mesh2	6/24/2020 11:31:06 PM	dc899f09...
#3: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k2_mesh3	6/24/2020 11:31:21 PM	66cce599...
#4: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k2_mesh4	6/24/2020 11:31:44 PM	d806a711...
#5: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k2_mesh5	6/24/2020 11:31:53 PM	35cbee2e...
#6: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta5_cAstat120_ConvStudy2_k3_mesh0	6/24/2020 11:32:16 PM	3912483f...
#7: CLpaper_StaticDropletOnSlipWall	StaticDropletOnWall_90Deg_beta5_cAstat12

In [10]:
NUnit.Framework.Assert.AreEqual(72, evalSess.Count, $"Not enough sessions for evaluation.");

In [11]:
public static Plot2Ddata[] GetConvergencePlotData(List<ISessionInfo> dataSess, string study) {

    string[] FieldsToCompare = new string[] {"VelocityX", "VelocityY", "Pressure"};
    int[] pOrder = new int[] { 2, 3, 4 };
    PointTypes[] ptTypes = new PointTypes[] { PointTypes.Circle, PointTypes.LowerTriangle, PointTypes.Box };

    Plot2Ddata[] ConvPlts = new Plot2Ddata[2];

    Plot2Ddata vPlt = new Plot2Ddata().WithLogX().WithLogY();
    vPlt.Xlabel = "h";
    vPlt.Ylabel = "L2-error velocities";
    vPlt.LegendAlignment = new string[] { "i", "b", "r" };
    ConvPlts[0] = vPlt;

    Plot2Ddata pPlt = new Plot2Ddata().WithLogX().WithLogY();
    pPlt.Xlabel = "h";
    pPlt.Ylabel = "L2-error pressure";
    pPlt.LegendAlignment = new string[] { "i", "b", "r" };
    ConvPlts[1] = pPlt;

    int lcIdx = 0;
    foreach(int p in pOrder) {
        var pDegStudySess = dataSess.Where(s => s.Name.Contains(study) 
                                                && Convert.ToInt32(s.KeysAndQueries["DGdegree:Velocity*"]) == p).ToArray();
        
        ITimestepInfo[] timesteps = pDegStudySess.Select(s => s.Timesteps.Newest()).ToArray();
        
        double[] GridRes;
        Dictionary<string, long[]> __DOFs;
        Dictionary<string, double[]> L2Errors;
        Guid[] timestepIds;

        DGFieldComparison.ComputeErrors(FieldsToCompare, timesteps, out GridRes, out __DOFs, out L2Errors, out timestepIds, NormType.L2_embedded);

        var fmt = new PlotFormat();
        fmt.Style = Styles.LinesPoints;
        fmt.DashType = DashTypes.Dashed;
        fmt.PointType = ptTypes[lcIdx];
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(lcIdx + 1); //LineColors.Blue;
        
        // velocities
        ConvPlts[0].AddDataGroup($"k = {p}", GridRes, L2Errors["VelocityX"].Zip(L2Errors["VelocityY"], (x,y) => Math.Sqrt(x*x + y*y)).ToArray(), fmt);

        // pressure
        ConvPlts[1].AddDataGroup($"k' = {p-1}", GridRes, L2Errors["Pressure"], fmt);

        lcIdx++;
    }

    return ConvPlts;    
    
}

In [12]:
public static void CheckOrderOfConvergence(Plot2Ddata[] convPlts, double[] refEOCvelocities, double[] refEOCpressure) {

    // velocities 
    Console.WriteLine($"EOC || u - u_ref ||_2");
    var slopes = convPlts[0].Regression();
    for (int i = 0; i < slopes.Count(); i++) {
        string order = slopes.Pick(i).Key;
        double value = slopes.Pick(i).Value;
        Console.WriteLine($"{order}: {value}");
        NUnit.Framework.Assert.AreEqual(refEOCvelocities[i], value, 0.05, 
            $"l2-error for {order} does not match reference value {refEOCvelocities[i]}.");
    }

    // pressure
    Console.WriteLine($"EOC || p - p_ref ||_2");
    slopes = convPlts[1].Regression();
    for (int i = 0; i < slopes.Count(); i++) {
        string order = slopes.Pick(i).Key;
        double value = slopes.Pick(i).Value;
        Console.WriteLine($"{order}: {value}");
        NUnit.Framework.Assert.AreEqual(refEOCpressure[i], value, 0.05, 
            $"l2-error for {order} does not match reference value {refEOCpressure[i]}.");
    }
} 

In [13]:
var ConvPlt_b5_c120 = GetConvergencePlotData(evalSess, "beta5_cAstat120_ConvStudy");
var ConvPlt_b0_c120 = GetConvergencePlotData(evalSess, "beta0_cAstat120_ConvStudy");
var ConvPlt_b5_c90 = GetConvergencePlotData(evalSess, "beta5_cAstat90_ConvStudy");
var ConvPlt_b0_c90 = GetConvergencePlotData(evalSess, "beta0_cAstat90_ConvStudy");

### Table 1

EOC for $\theta_{stat} = 120°$ and $\beta_S = 5$

In [14]:
CheckOrderOfConvergence(ConvPlt_b5_c120, new double[] { 0.7, 0.9, 1.1 }, new double[] { 0.1, 0.1, 0.1 });

EOC || u - u_ref ||_2
k = 2: 0.712997388525173
k = 3: 0.9137886378090431
k = 4: 1.0851746214938711
EOC || p - p_ref ||_2
k' = 1: 0.10361318832839984
k' = 2: 0.08999457445354776
k' = 3: 0.09070411829359512


EOC for $\theta_{stat} = 120°$ and $\beta_S = 0$

In [15]:
CheckOrderOfConvergence(ConvPlt_b0_c120, new double[] { 1.1, 1.1, 1.1 }, new double[] { 0.2, 0.1, 0.1 });

EOC || u - u_ref ||_2
k = 2: 1.0556693877516163
k = 3: 1.1260008027041315
k = 4: 1.1418969326067874
EOC || p - p_ref ||_2
k' = 1: 0.1779907565590018
k' = 2: 0.10506652633178055
k' = 3: 0.08775921852114207


EOC for $\theta_{stat} = 90°$ and $\beta_S = 5$

In [16]:
CheckOrderOfConvergence(ConvPlt_b5_c90, new double[] { 1.8, 1.7, 1.7 }, new double[] { 0.9, 0.8, 0.8 });

EOC || u - u_ref ||_2
k = 2: 1.7958220554107838
k = 3: 1.7099465476342082
k = 4: 1.7059157296013703
EOC || p - p_ref ||_2
k' = 1: 0.9006196076315468
k' = 2: 0.8102713958414136
k' = 3: 0.8366108074593624


EOC for $\theta_{stat} = 90°$ and $\beta_S = 0$

In [17]:
CheckOrderOfConvergence(ConvPlt_b0_c90, new double[] { 2.8, 3.8, 4.5 }, new double[] { 1.9, 2.6, 2.8 });

EOC || u - u_ref ||_2
k = 2: 2.782185308868151
k = 3: 3.8054671054702975
k = 4: 4.530585417453844
EOC || p - p_ref ||_2
k' = 1: 1.949962497231718
k' = 2: 2.584637495234956
k' = 3: 2.8391053921904796


### Figure B1

In [18]:
Plot2Ddata[,] FigB1 = new Plot2Ddata[2, 2];

var fig = ConvPlt_b5_c120[0];
fig.YrangeMin = 1e-4;
fig.YrangeMax = 1e-2;
FigB1[0, 0] = fig;

fig = ConvPlt_b5_c120[1];
fig.YrangeMin = 0.05;
fig.YrangeMax = 0.1;
FigB1[0, 1] = fig;

fig = ConvPlt_b0_c120[0];
fig.YrangeMin = 1e-4;
fig.YrangeMax = 2e-2;
FigB1[1, 0] = fig;

fig = ConvPlt_b0_c120[1];
fig.YrangeMin = 0.05;
fig.YrangeMax = 0.11;
FigB1[1, 1] = fig;

FigB1.ToGnuplot().PlotSVG(xRes:1200,yRes:900)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 -4 
 
 
 
 
 10 -3 
 
 
 
 
 10 -2 
 
 
 
 
 10 -2 
 
 
 
 
 10 -1 
 
 
 
 
 10 0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 L2-error velocities 
 
 
 
 
 h 
 
 
 
 
 k = 2 
 
 
 
 
 k = 2 
 
 
 
 
 
 
 
 
 
 
 
 k = 3 
 
 
 k = 3 
 
 
 
 
 
 
 
 
 
 
 
 k = 4 
 
 
 k = 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 -1 
 
 
 
 
 10 -2 
 
 
 
 
 10 -1 
 
 
 
	<path stroke='black' d='M978.2,388.0 L978.2,383.5 M978.2,40.6 L978.2,45.1 M1020.0,388.0 L1020.0,383.5 M1020.0,40.6 L1020.0,45.1
 M1049.7,388.0 L1049.7,383.5 M1049.7,40.6 L1049.7,45.1 M1072.7,388.0 L1072.7,383.5 M1072.7,40.6 L1072.7,45.1
 M1091.5,388.0 L1091.5,383.5 M1091.5,40.6 L1091.5,45.1 M1107.3,388.0 L1107.3,383.5 M1107.3,40.6 L1107.3,45.1
 M1121.1,388.0 L1121.1,383.5 M1121.1,40.6 L1121.1,45.1 M1133.2,388.0 L1133.2,383.5 M1133.2,40.6 L1133.2,45.1
 M1144.1,388.0 L1144.1,379.0 M1144.1,40.6 L1144.1,49.6 '/> 
 10 0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 L2-error pressure 
 
 
 
 
 h 
 
 
 
 
 k' = 1 
 
 
 
 
 k' = 1 
 
 
 
 
 
 
 
 
 
 
 
 k' = 2 
 
 
 k' = 2 
 
 
 
 
 
 
 
 
 
 
 
 k' = 3 
 
 
 k' = 3 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 -4 
 
 
 
 
 10 -3 
 
 
 
 
 10 -2 
 
 
 
 
 10 -2 
 
 
 
 
 10 -1 
 
 
 
 
 10 0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 L2-error velocities 
 
 
 
 
 h 
 
 
 
 
 k = 2 
 
 
 
 
 k = 2 
 
 
 
 
 
 
 
 
 
 
 
 k = 3 
 
 
 k = 3 
 
 
 
 
 
 
 
 
 
 
 
 k = 4 
 
 
 k = 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 -1 
 
 
 
 
 10 -2 
 
 
 
 
 10 -1 
 
 
 
	<path stroke='black' d='M978.2,838.0 L978.2,833.5 M978.2,490.5 L978.2,495.0 M1020.0,838.0 L1020.0,833.5 M1020.0,490.5 L1020.0,495.0
 M1049.7,838.0 L1049.7,833.5 M1049.7,490.5 L1049.7,495.0 M1072.7,838.0 L1072.7,833.5 M1072.7,490.5 L1072.7,495.0
 M1091.5,838.0 L1091.5,833.5 M1091.5,490.5 L1091.5,495.0 M1107.3,838.0 L1107.3,833.5 M1107.3,490.5 L1107.3,495.0
 M1121.1,838.0 L1121.1,833.5 M1121.1,490.5 L1121.1,495.0 M1133.2,838.0 L1133.2,833.5 M1133.2,490.5 L1133.2,495.0
 M1144.1,838.0 L1144.1,829.0 M1144.1,490.5 L1144.1,499.5 '/> 
 10 0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 L2-error pressure 
 
 
 
 
 h 
 
 
 
 
 k' = 1 
 
 
 
 
 k' = 1 
 
 
 
 
 
 
 
 
 
 
 
 k' = 2 
 
 
 k' = 2 
 
 
 
 
 
 
 
 
 
 
 
 k' = 3 
 
 
 k' = 3

### Figure B2

In [19]:
Plot2Ddata[,] FigB2 = new Plot2Ddata[2, 2];

var fig = ConvPlt_b5_c90[0];
fig.YrangeMin = 1e-7;
fig.YrangeMax = 1e-3;
FigB2[0, 0] = fig;

fig = ConvPlt_b5_c90[1];
fig.YrangeMin = 1e-5;
fig.YrangeMax = 1e-2;
FigB2[0, 1] = fig;

fig = ConvPlt_b0_c90[0];
fig.YrangeMin = 1e-12;
fig.YrangeMax = 1e-2;
FigB2[1, 0] = fig;

fig = ConvPlt_b0_c90[1];
fig.YrangeMin = 1e-8;
fig.YrangeMax = 1e-2;
FigB2[1, 1] = fig;

FigB2.ToGnuplot().PlotSVG(xRes:1200,yRes:900)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 -7 
 
 
 
 
 10 -6 
 
 
 
 
 10 -5 
 
 
 
 
 10 -4 
 
 
 
 
 10 -3 
 
 
 
 
 10 -2 
 
 
 
 
 10 -1 
 
 
 
 
 10 0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 L2-error velocities 
 
 
 
 
 h 
 
 
 
 
 k = 2 
 
 
 
 
 k = 2 
 
 
 
 
 
 
 
 
 
 
 
 k = 3 
 
 
 k = 3 
 
 
 
 
 
 
 
 
 
 
 
 k = 4 
 
 
 k = 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 -5 
 
 
 
	<path stroke='black' d='M669.5,353.1 L674.0,353.1 M1144.1,353.1 L1139.6,353.1 M669.5,332.7 L674.0,332.7 M1144.1,332.7 L1139.6,332.7
 M669.5,318.3 L674.0,318.3 M1144.1,318.3 L1139.6,318.3 M669.5,307.1 L674.0,307.1 M1144.1,307.1 L1139.6,307.1
 M669.5,297.9 L674.0,297.9 M1144.1,297.9 L1139.6,297.9 M669.5,290.1 L674.0,290.1 M1144.1,290.1 L1139.6,290.1
 M669.5,283.4 L674.0,283.4 M1144.1,283.4 L1139.6,283.4 M669.5,277.5 L674.0,277.5 M1144.1,277.5 L1139.6,277.5
 M669.5,272.2 L678.5,272.2 M1144.1,272.2 L1135.1,272.2 '/> 
 10 -4 
 
 
 
	<path stroke='black' d='M669.5,237.3 L674.0,237.3 M1144.1,237.3 L1139.6,237.3 M669.5,216.9 L674.0,216.9 M1144.1,216.9 L1139.6,216.9
 M669.5,202.5 L674.0,202.5 M1144.1,202.5 L1139.6,202.5 M669.5,191.3 L674.0,191.3 M1144.1,191.3 L1139.6,191.3
 M669.5,182.1 L674.0,182.1 M1144.1,182.1 L1139.6,182.1 M669.5,174.3 L674.0,174.3 M1144.1,174.3 L1139.6,174.3
 M669.5,167.6 L674.0,167.6 M1144.1,167.6 L1139.6,167.6 M669.5,161.7 L674.0,161.7 M1144.1,161.7 L1139.6,161.7
 M669.5,156.4 L678.5,156.4 M1144.1,156.4 L1135.1,156.4 '/> 
 10 -3 
 
 
 
 
 10 -2 
 
 
 
 
 10 -2 
 
 
 
 
 10 -1 
 
 
 
	<path stroke='black' d='M978.2,388.0 L978.2,383.5 M978.2,40.6 L978.2,45.1 M1020.0,388.0 L1020.0,383.5 M1020.0,40.6 L1020.0,45.1
 M1049.7,388.0 L1049.7,383.5 M1049.7,40.6 L1049.7,45.1 M1072.7,388.0 L1072.7,383.5 M1072.7,40.6 L1072.7,45.1
 M1091.5,388.0 L1091.5,383.5 M1091.5,40.6 L1091.5,45.1 M1107.3,388.0 L1107.3,383.5 M1107.3,40.6 L1107.3,45.1
 M1121.1,388.0 L1121.1,383.5 M1121.1,40.6 L1121.1,45.1 M1133.2,388.0 L1133.2,383.5 M1133.2,40.6 L1133.2,45.1
 M1144.1,388.0 L1144.1,379.0 M1144.1,40.6 L1144.1,49.6 '/> 
 10 0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 L2-error pressure 
 
 
 
 
 h 
 
 
 
 
 k' = 1 
 
 
 
 
 k' = 1 
 
 
 
 
 
 
 
 
 
 
 
 k' = 2 
 
 
 k' = 2 
 
 
 
 
 
 
 
 
 
 
 
 k' = 3 
 
 
 k' = 3 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 -12 
 
 
 
 
 10 -11 
 
 
 
 
 10 -10 
 
 
 
 
 10 -9 
 
 
 
 
 10 -8 
 
 
 
 
 10 -7 
 
 
 
 
 10 -6 
 
 
 
 
 10 -5 
 
 
 
 
 10 -4 
 
 
 
 
 10 -3 
 
 
 
 
 10 -2 
 
 
 
 
 10 -2 
 
 
 
 
 10 -1 
 
 
 
 
 10 0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 L2-error velocities 
 
 
 
 
 h 
 
 
 
 
 k = 2 
 
 
 
 
 k = 2 
 
 
 
 
 
 
 
 
 
 
 
 k = 3 
 
 
 k = 3 
 
 
 
 
 
 
 
 
 
 
 
 k = 4 
 
 
 k = 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 -8 
 
 
 
 
 10 -7 
 
 
 
 
 10 -6 
 
 
 
 
 10 -5 
 
 
 
 
 10 -4 
 
 
 
 
 10 -3 
 
 
 
 
 10 -2 
 
 
 
 
 10 -2 
 
 
 
 
 10 -1 
 
 
 
	<path stroke='black' d='M978.2,838.0 L978.2,833.5 M978.2,490.5 L978.2,495.0 M1020.0,838.0 L1020.0,833.5 M1020.0,490.5 L1020.0,495.0
 M1049.7,838.0 L1049.7,833.5 M1049.7,490.5 L1049.7,495.0 M1072.7,838.0 L1072.7,833.5 M1072.7,490.5 L1072.7,495.0
 M1091.5,838.0 L1091.5,833.5 M1091.5,490.5 L1091.5,495.0 M1107.3,838.0 L1107.3,833.5 M1107.3,490.5 L1107.3,495.0
 M1121.1,838.0 L1121.1,833.5 M1121.1,490.5 L1121.1,495.0 M1133.2,83